#### MySQL Connection

In [1]:
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False

import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine.url import URL

load_dotenv()

url = URL.create("mysql+pymysql",
                 username="root",
                 password=os.getenv('MYSQL_PW'),
                 host="localhost")
engine = create_engine(url)
%sql engine

# ON DELETE CASCADE
#### ON DELETE CASCADE constraint in SQL automatically deletes rows from a child table when the corresponding referenced rows in the parent table are deleted. It prevents "orphaned" records and ensures referential integrity across related database tables without requiring manual, multi-step delete queries.

#### If we try to delete a row from customers table, we will get an error since the orders table is using the id of the customers table as foreign key constraint

In [9]:
%%sql

USE blinkit;

DELETE FROM customers
WHERE last_name = 'George'

RuntimeError: (pymysql.err.IntegrityError) (1451, 'Cannot delete or update a parent row: a foreign key constraint fails (`blinkit`.`orders`, CONSTRAINT `orders_ibfk_1` FOREIGN KEY (`customer_id`) REFERENCES `customers` (`id`))')
[SQL: DELETE FROM customers
WHERE last_name = 'George']
(Background on this error at: https://sqlalche.me/e/20/gkpj)


In [10]:
%%sql

USE blinkit;

DELETE FROM customers
WHERE id = 1

RuntimeError: (pymysql.err.IntegrityError) (1451, 'Cannot delete or update a parent row: a foreign key constraint fails (`blinkit`.`orders`, CONSTRAINT `orders_ibfk_1` FOREIGN KEY (`customer_id`) REFERENCES `customers` (`id`))')
[SQL: DELETE FROM customers
WHERE id = 1]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


In [11]:
%%sql

DROP TABLE orders

""


## Now we will add the ON DELETE CASCADE constraint to the orders table

In [12]:
%%sql

CREATE TABLE orders
    (
        id INT AUTO_INCREMENT PRIMARY KEY,
        order_date DATE,
        amount DECIMAL(8,2),
        customer_id INT,
        FOREIGN KEY (customer_id) REFERENCES customers(id) ON DELETE CASCADE
    );

""


In [13]:
%%sql

INSERT INTO orders (order_date, amount, customer_id)
VALUES ('2016-02-10', 99.99, 1),
       ('2017-11-11', 35.50, 1),
       ('2014-12-12', 800.67, 2),
       ('2015-01-03', 12.50, 2),
       ('1999-04-11', 450.25, 5);

""


#### Now we will try to delete a row from customers again

In [14]:
%%sql

USE blinkit;

DELETE FROM customers
WHERE last_name = 'George'

""


In [15]:
%%sql

SELECT *
FROM customers

,id,first_name,last_name,email
0,2,George,Michael,gm@gmail.com
1,3,David,Bowie,david@gmail.com
2,4,Blue,Steele,blue@gmail.com
3,5,Bette,Davis,bette@aol.com


In [16]:
%%sql

SELECT *
FROM orders

,id,order_date,amount,customer_id
0,3,2014-12-12,800.67,2
1,4,2015-01-03,12.50,2
2,5,1999-04-11,450.25,5
